# Customer Bronze to Silver

This notebook takes raw customer data from the Bronze layer, cleans and validates it, and prepares it for the Silver layer.

At a high level, it reads customer records, removes bad or duplicate data, stores rejected records in an exception table, and upserts the clean records into the Silver customer table.

The goal is to make customer data ready for downstream reporting and business use.

In [0]:
import logging
import sys
from pyspark.sql.functions import col, lit, trim, current_timestamp, current_date, monotonically_increasing_id, when

# ==========================================
# 1. LOGGING & INFRASTRUCTURE CONFIGURATION
# ==========================================
logger = logging.getLogger("ERP_Silver_Transformations")
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter('%(asctime)s - [%(levelname)s] - %(message)s'))
    logger.addHandler(handler)



In [0]:
# Enterprise Decoupled Storage Routing
STORAGE_ACCOUNT = "bbmanufacturingprod"
CONTAINER_NAME = "silver"
BASE_ADLS_PATH = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

# Target Schemas and Locations
CATALOG_NAME='erp_lakehouse'
SILVER_SCHEMA = "silver"
SILVER_TARGET_PATH = f"{BASE_ADLS_PATH}/delta/sl_customer"
EXCEPTION_TARGET_PATH = f"{BASE_ADLS_PATH}/delta/exception_table"

# Create the dedicated silver schema isolation layer
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SILVER_SCHEMA}")

In [0]:
# Initialize Target Tables with explicit locations if they don't exist
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.{SILVER_SCHEMA}.sl_Customer (
    CompanyKey STRING, BrandKey STRING, CustSrcId STRING, CustContactSrcId STRING,
    SalesRepInSrcId STRING, CustBillToAddrSrcId STRING, CustShipToAddrSrcId STRING,
    CustKey STRING, CustName STRING, CustAltName1 STRING, CustAltName2 STRING,
    CustContactName STRING, AddrLn1 STRING, AddrLn2 STRING, City STRING, State STRING,
    Country STRING, ZipCd STRING, TelPhNo1 STRING, EmailAddr STRING,
    SourceUpdatedTime TIMESTAMP, SysCreatedTime TIMESTAMP, RecordStatus STRING
) USING DELTA LOCATION '{SILVER_TARGET_PATH}'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.{SILVER_SCHEMA}.exception_table (
    ExceptionID BIGINT, BrandKey STRING, RecordKey STRING, TableName STRING,
    ColumnName STRING, ExceptionDetails STRING, SysCreatedDate DATE, SysCreatedBy STRING
) USING DELTA LOCATION '{EXCEPTION_TARGET_PATH}'
""")


In [0]:
# ==========================================
# 2. HIGH-PERFORMANCE PIPELINE PROCESSING
# ==========================================
logger.info("📥 Extracting change-set data from Bronze layer...")
bronze_df = spark.read.table("erp_lakehouse.bronze.bz_customercard")

# Standardize baseline structural keys upfront
enriched_df = bronze_df.withColumn("CompanyKey", lit("ABC")) \
                        .withColumn("BrandKey", lit("ABC")) \
                        .withColumn("RecordStatus", lit("0"))

# --- OPTIMIZED DQ STRATEGY (Eliminating the expensive Window Shuffle) ---
# Route 1: Capture Nulls or Blanks immediately via lightweight filter
null_condition = (col("No").isNull()) | (trim(col("No")) == "")
null_records_df = enriched_df.filter(null_condition)

# Route 2: In-Memory Deduplication using dropDuplicates 
# This leverages Spark's internal hash-aggregate instead of a heavy sort-merge shuffle
valid_pkeys_df = enriched_df.filter(~null_condition)
deduped_records_df = valid_pkeys_df.dropDuplicates(["No", "CompanyKey", "BrandKey"])

# Route 3: Isolate inline duplicates by subtracting the deduped dataset from the valid population
duplicate_records_df = valid_pkeys_df.subtract(deduped_records_df)


In [0]:
# ==========================================
# 3. ROUTE LOGICAL EXCEPTIONS TO AUDIT LAYER
# ==========================================
logger.info("⚠️ Formatting exceptions for ingestion into the DQ Audit layer...")

exception_df_1 = duplicate_records_df.withColumn("ExceptionID", monotonically_increasing_id()) \
    .select(
        col("ExceptionID").cast("bigint"),
        lit("ABC").alias("BrandKey"),
        col("No").cast("string").alias("RecordKey"),
        lit("bz_CustomerCard").alias("TableName"),
        lit("No").alias("ColumnName"),
        lit("Duplicate record found inside micro-batch").alias("ExceptionDetails"),
        current_date().alias("SysCreatedDate"),
        lit("system_spark_engine").alias("SysCreatedBy")
    )

exception_df_2 = null_records_df.withColumn("ExceptionID", monotonically_increasing_id()) \
    .select(
        col("ExceptionID").cast("bigint"),
        lit("ABC").alias("BrandKey"),
        lit("UNKNOWN_KEY").alias("RecordKey"),
        lit("bz_CustomerCard").alias("TableName"),
        lit("No").alias("ColumnName"),
        lit("Null or Blank Primary Key drop").alias("ExceptionDetails"),
        current_date().alias("SysCreatedDate"),
        lit("system_spark_engine").alias("SysCreatedBy")
    )

final_exceptions = exception_df_1.unionByName(exception_df_2)

if not final_exceptions.isEmpty():
    logger.warning(f"💾 Writing corrupted records to exception table path: {EXCEPTION_TARGET_PATH}")
    final_exceptions.write.format("delta").mode("append").save(EXCEPTION_TARGET_PATH)


In [0]:

# ==========================================
# 4. STRUCTURE CLEAN SILVER LAYER VIEW
# ==========================================
logger.info("✨ Transforming clean data for Silver presentation layer...")

from pyspark.sql.functions import coalesce, lit

# Use cleanly optimized coalesce handling expressions

final_df = deduped_records_df.select(
    col("CompanyKey").cast("string"),
    col("BrandKey").cast("string"),
    col("No").alias("CustSrcId").cast("string"),
    coalesce(col("Primary_Contact_No"), lit("")).alias("CustContactSrcId").cast("string"),
    coalesce(col("Salesperson_Code"), lit("")).alias("SalesRepInSrcId").cast("string"),
    coalesce(col("Bill_to_Customer_No"), lit("")).alias("CustBillToAddrSrcId").cast("string"),
    coalesce(col("Ship_to_Code"), lit("")).alias("CustShipToAddrSrcId").cast("string"),
    coalesce(col("No"), lit("")).alias("CustKey").cast("string"),
    coalesce(col("Name"), lit("")).alias("CustName").cast("string"),
    coalesce(col("Search_Name"), lit("")).alias("CustAltName1").cast("string"),
    coalesce(col("Name_2"), lit("")).alias("CustAltName2").cast("string"),
    coalesce(col("ContactName"), lit("")).alias("CustContactName").cast("string"),
    coalesce(col("Address"), lit("")).alias("AddrLn1").cast("string"),
    coalesce(col("Address_2"), lit("")).alias("AddrLn2").cast("string"),
    coalesce(col("City"), lit("")).alias("City").cast("string"),
    coalesce(col("County"), lit("")).alias("State").cast("string"),
    coalesce(col("Country_Region_Code"), lit("")).alias("Country").cast("string"),
    coalesce(col("Post_Code"), lit("")).alias("ZipCd").cast("string"),
    coalesce(col("Phone_No"), lit("")).alias("TelPhNo1").cast("string"),
    coalesce(col("E_Mail"), lit("")).alias("EmailAddr").cast("string"),
    col("Last_Date_Modified").alias("SourceUpdatedTime").cast("timestamp"),
    col("ingestion_time").alias("SysCreatedTime").cast("timestamp"),
    col("RecordStatus").cast("string")
)



In [0]:
# ==========================================
# 5. ATOMIC DELTA UPSERT OPERATION
# ==========================================
logger.info(f"🔄 Executing high-performance Delta Merge upsert into {SILVER_SCHEMA}.sl_Customer...")
final_df.createOrReplaceTempView("src_silver_changeset")

spark.sql(f"""
MERGE INTO {CATALOG_NAME}.{SILVER_SCHEMA}.sl_Customer AS tgt
USING src_silver_changeset AS src
ON tgt.CompanyKey = src.CompanyKey
  AND tgt.BrandKey = src.BrandKey
  AND tgt.CustSrcId = src.CustSrcId
WHEN MATCHED THEN
  UPDATE SET
    tgt.CustName = src.CustName,
    tgt.CustAltName1 = src.CustAltName1,
    tgt.CustAltName2 = src.CustAltName2,
    tgt.CustContactName = src.CustContactName,
    tgt.AddrLn1 = src.AddrLn1,
    tgt.AddrLn2 = src.AddrLn2,
    tgt.City = src.City,
    tgt.State = src.State,
    tgt.Country = src.Country,
    tgt.ZipCd = src.ZipCd,
    tgt.TelPhNo1 = src.TelPhNo1,
    tgt.EmailAddr = src.EmailAddr,
    tgt.SourceUpdatedTime = src.SourceUpdatedTime
WHEN NOT MATCHED THEN
  INSERT (
    CompanyKey, BrandKey, CustSrcId, CustContactSrcId, SalesRepInSrcId, CustBillToAddrSrcId, CustShipToAddrSrcId,
    CustKey, CustName, CustAltName1, CustAltName2, CustContactName, AddrLn1, AddrLn2, City, State, Country, ZipCd,
    TelPhNo1, EmailAddr, SourceUpdatedTime, SysCreatedTime, RecordStatus
  )
  VALUES (
    src.CompanyKey, src.BrandKey, src.CustSrcId, src.CustContactSrcId, src.SalesRepInSrcId, src.CustBillToAddrSrcId, src.CustShipToAddrSrcId,
    src.CustKey, src.CustName, src.CustAltName1, src.CustAltName2, src.CustContactName, src.AddrLn1, src.AddrLn2, src.City, src.State, src.Country, src.ZipCd,
    src.TelPhNo1, src.EmailAddr, src.SourceUpdatedTime, src.SysCreatedTime, src.RecordStatus
  )
""")

logger.info("🎉 Silver sync completed successfully.")